# Notebook 2: Preprocessing & Feature Engineering (Regression)

## California Housing Prices

**Goal**: Clean the data and prepare it for modeling.

Data Science Lifecycle:
1. Data Loading ✅
2. Exploratory Data Analysis ✅
3. **Preprocessing & Feature Engineering** ← we are here
4. Model Training & Evaluation
5. Model Comparison & Insights

---

### What we learned from EDA:
- MedInc is the strongest predictor
- Outliers in AveRooms, AveBedrms, AveOccup, Population
- Target is capped at 5.0
- Some features are skewed
- No missing values

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

Libraries loaded!


In [2]:
# load the data
df = pd.read_csv('../data/california_housing.csv')
print(f'Original shape: {df.shape}')
df.head()

Original shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 1. Handling Outliers

From EDA, we noticed some extreme values. Let's handle them.

**Strategy**: We'll cap outliers at the 99th percentile. This is a common approach —
we don't want to remove too many rows, but we also don't want extreme values
to mess up our models.

In [3]:
# columns with outliers (from our EDA)
outlier_cols = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']

print('Before capping:')
for col in outlier_cols:
    print(f'  {col}: max = {df[col].max():.2f}, 99th percentile = {df[col].quantile(0.99):.2f}')

# cap at 99th percentile
for col in outlier_cols:
    upper_limit = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=upper_limit)

print('\nAfter capping:')
for col in outlier_cols:
    print(f'  {col}: max = {df[col].max():.2f}')

Before capping:
  AveRooms: max = 141.91, 99th percentile = 10.36
  AveBedrms: max = 34.07, 99th percentile = 2.13
  AveOccup: max = 1243.33, 99th percentile = 5.39
  Population: max = 35682.00, 99th percentile = 5805.83

After capping:
  AveRooms: max = 10.36
  AveBedrms: max = 2.13
  AveOccup: max = 5.39
  Population: max = 5805.83


## 2. Handling the Capped Target

Our target (MedHouseVal) is capped at 5.0. We have two choices:
1. Remove those rows (we lose some data)
2. Keep them (our model might learn the cap instead of the real pattern)

Let's remove them — it's a small percentage and gives us cleaner data.

In [4]:
# how many rows are capped?
capped = (df['MedHouseVal'] >= 5.0).sum()
print(f'Rows with capped target: {capped} ({capped/len(df)*100:.1f}%)')

# remove them
df = df[df['MedHouseVal'] < 5.0].copy()
print(f'Shape after removing capped values: {df.shape}')

Rows with capped target: 992 (4.8%)
Shape after removing capped values: (19648, 9)


## 3. Feature Engineering

Let's create some new features that might help our models.
Feature engineering is about using domain knowledge to create
more meaningful inputs for the model.

In [5]:
# create new features

# bedroom ratio: what fraction of rooms are bedrooms?
df['BedroomRatio'] = df['AveBedrms'] / df['AveRooms']

# rooms per person: how crowded is the area?
df['RoomsPerPerson'] = df['AveRooms'] / df['AveOccup']

print('New features created!')
print(f'New shape: {df.shape}')
df[['AveRooms', 'AveBedrms', 'AveOccup', 'BedroomRatio', 'RoomsPerPerson']].head()

New features created!
New shape: (19648, 11)


,AveRooms,AveBedrms,AveOccup,BedroomRatio,RoomsPerPerson
0,6.984127,1.023810,2.555556,0.146591,2.732919
1,6.238137,0.971880,2.109842,0.155797,2.956685
2,8.288136,1.073446,2.802260,0.129516,2.957661
3,5.817352,1.073059,2.547945,0.184458,2.283154
4,6.281853,1.081081,2.181467,0.172096,2.879646


## 4. Train-Test Split

**Why split?** We need to test our model on data it has never seen before.
If we train and test on the same data, we can't tell if the model actually
learned patterns or just memorized the answers.

We'll use 80% for training and 20% for testing.

In [6]:
# separate features (X) and target (y)
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')

Features shape: (19648, 10)
Target shape: (19648,)

Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'BedroomRatio', 'RoomsPerPerson']


In [7]:
# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTrain/Test ratio: {X_train.shape[0]/len(X)*100:.0f}% / {X_test.shape[0]/len(X)*100:.0f}%')

Training set: 15718 samples
Test set: 3930 samples

Train/Test ratio: 80% / 20%


## 5. Feature Scaling

**Why scale?** Some algorithms (like Linear Regression, SVM, KNN) are sensitive
to the scale of features. If one feature ranges from 0-1 and another from 0-10000,
the model might think the bigger feature is more important.

**StandardScaler** transforms each feature to have mean=0 and std=1.

**Important**: We fit the scaler on the training data only, then transform both
train and test. This prevents "data leakage" — the test set should be completely unseen.

In [8]:
# scale the features
scaler = StandardScaler()

# fit on training data, transform both
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

print('Before scaling (first 3 rows of training data):')
print(X_train.head(3).round(2))
print('\nAfter scaling (first 3 rows of training data):')
print(X_train_scaled.head(3).round(2))

Before scaling (first 3 rows of training data):
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
12794    1.66      20.0      4.62       0.98      1287.0      3.92     38.64   
14570    3.45      35.0      4.42       0.98      1340.0      2.62     32.83   
9328     3.99      35.0      4.61       0.98       413.0      2.10     37.96   

       Longitude  BedroomRatio  RoomsPerPerson  
12794    -121.46          0.21            1.18  
14570    -117.21          0.22            1.69  
9328     -122.53          0.21            2.20  

After scaling (first 3 rows of training data):
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
12794   -1.28     -0.67     -0.50      -0.57       -0.13      1.35      1.38   
14570   -0.15      0.52     -0.66      -0.60       -0.08     -0.43     -1.32   
9328     0.20      0.52     -0.52      -0.57       -1.02     -1.15      1.07   

       Longitude  BedroomRatio  RoomsPerPerson  
12794      -0.94 

In [9]:
# verify: scaled features should have mean ≈ 0 and std ≈ 1
print('Scaled training data statistics:')
print(f'Mean (should be ~0):\n{X_train_scaled.mean().round(4)}')
print(f'\nStd (should be ~1):\n{X_train_scaled.std().round(4)}')

Scaled training data statistics:
Mean (should be ~0):
MedInc           -0.0
HouseAge          0.0
AveRooms          0.0
AveBedrms        -0.0
Population       -0.0
AveOccup          0.0
Latitude         -0.0
Longitude         0.0
BedroomRatio     -0.0
RoomsPerPerson    0.0
dtype: float64

Std (should be ~1):
MedInc            1.0
HouseAge          1.0
AveRooms          1.0
AveBedrms         1.0
Population        1.0
AveOccup          1.0
Latitude          1.0
Longitude         1.0
BedroomRatio      1.0
RoomsPerPerson    1.0
dtype: float64


## 6. Save Preprocessed Data

Let's save everything so we can use it directly in the modeling notebook.

In [10]:
# save preprocessed data for the next notebook
X_train_scaled.to_csv('../data/X_train_reg.csv', index=False)
X_test_scaled.to_csv('../data/X_test_reg.csv', index=False)
y_train.to_csv('../data/y_train_reg.csv', index=False)
y_test.to_csv('../data/y_test_reg.csv', index=False)

# also save unscaled versions (some models like trees don't need scaling)
X_train.to_csv('../data/X_train_reg_unscaled.csv', index=False)
X_test.to_csv('../data/X_test_reg_unscaled.csv', index=False)

print('All preprocessed data saved to data/ folder!')
print(f'  X_train: {X_train_scaled.shape}')
print(f'  X_test: {X_test_scaled.shape}')
print(f'  y_train: {y_train.shape}')
print(f'  y_test: {y_test.shape}')

All preprocessed data saved to data/ folder!
  X_train: (15718, 10)
  X_test: (3930, 10)
  y_train: (15718,)
  y_test: (3930,)


## Summary

What we did in this notebook:

| Step | What | Why |
|------|------|-----|
| 1 | Capped outliers at 99th percentile | Prevent extreme values from distorting models |
| 2 | Removed capped target values | Cleaner target distribution for better learning |
| 3 | Created BedroomRatio and RoomsPerPerson | Domain-driven features that capture useful patterns |
| 4 | Train-test split (80/20) | Need unseen data to evaluate model performance |
| 5 | StandardScaler on features | Some algorithms need features on the same scale |

### Next Steps
In the next notebook, we'll train multiple regression models and compare their performance!